# Semantic Legal Domain Taxonomy — Gemma-2-9b-it

**Goal:** Build a content-based taxonomy of ~30 semantic legal domains that works for BOTH laws and BGE court decisions.

**Why semantic, not SR-number-based:**  
BGE court decisions have no SR numbers. We need a unified domain vocabulary derived from content, so the same query classifier can narrow the search across both corpora.

**Pipeline:**
1. Define ~30 semantic domain names (e.g. Vertragsrecht, Strafrecht, Umweltrecht)
2. For each unique law: Gemma reads title + Art. 1 → assigns to one domain
3. For each BGE case: Gemma reads first consideration text → assigns to one domain
4. Save law→domain and BGE→domain mappings
5. Validate coverage against gold citations

**At query time:** Gemma classifies the exam question → domain → search only that domain's articles (~1/30 of corpus)

## Section 1 — Setup

In [ ]:
!pip install -q bitsandbytes accelerate

In [ ]:
import os, re, time, json
import numpy as np
import pandas as pd
from collections import defaultdict
import torch
import warnings
warnings.filterwarnings('ignore')

In [ ]:
BASE   = '/kaggle/input/llm-agentic-legal-information-retrieval' if os.path.exists('/kaggle/input') else '../data'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['HF_TOKEN']    = secrets.get_secret('HUGGING_FACE_HUB_TOKEN')
os.environ['HF_USERNAME'] = 'nhkhoi91'

print('BASE:', BASE)
print('device:', DEVICE)

## Section 2 — Load Data & Extract Art. 1

In [ ]:
laws = pd.read_csv(f'{BASE}/laws_de.csv')
laws['text']  = laws['text'].fillna('')
laws['title'] = laws['title'].fillna('')
laws['law_id'] = laws['citation'].str.extract(r'(\S+)$')

court = pd.read_csv(f'{BASE}/court_considerations.csv')
court['text'] = court['text'].fillna('')

train = pd.read_csv(f'{BASE}/train.csv')

print(f'Laws: {len(laws):,} articles, {laws["law_id"].nunique():,} unique laws')
print(f'Court considerations: {len(court):,} rows')
print(f'Train queries: {len(train):,}')
print()
print('Court columns:', court.columns.tolist())
print(court.head(3))

In [ ]:
# Extract Art. 1 text for each unique law (best domain signal)
art1_rows = laws[laws['citation'].str.match(r'Art\.\s*1\s*(Abs\.\s*1\s*)?\S+$')].drop_duplicates('law_id')

# Build law metadata: one row per unique law
law_meta = laws.drop_duplicates('law_id')[['law_id', 'title']].copy()
law_meta['short_title'] = law_meta['title'].str.split(' - ').str[0].str.strip()
law_meta = law_meta.merge(
    art1_rows[['law_id', 'text']].rename(columns={'text': 'art1_text'}),
    on='law_id', how='left'
)
law_meta['art1_text'] = law_meta['art1_text'].fillna('')

print(f'Unique laws: {len(law_meta)}')
print(f'  with Art. 1 text: {(law_meta["art1_text"] != "").sum()}')
print(f'  title-only:       {(law_meta["art1_text"] == "").sum()}')

In [ ]:
# 30 semantic Swiss legal domains — the taxonomy vocabulary
# These are the domain names both the law classifier AND the query classifier will use.
DOMAIN_LIST = [
    "Verfassungsrecht",           # constitutional law, fundamental rights (BV, EMRK)
    "Völkerrecht",                # international law, treaties
    "Staatsorganisation",         # parliament, government structure, federal personnel
    "Verwaltungsrecht",           # general admin law, procedure, courts (VwVG, BGG)
    "Ausländerrecht",             # foreigners, asylum, citizenship (AIG, AsylG, BüG)
    "Politische_Rechte",          # elections, voting, political participation
    "Datenschutzrecht",           # data protection, privacy (DSG)
    "Zivilprozessrecht",          # civil procedure, jurisdiction (ZPO, IPRG)
    "Strafprozessrecht",          # criminal procedure (StPO, JStPO)
    "Schuldbetreibungsrecht",     # debt enforcement, bankruptcy (SchKG)
    "Familienrecht",              # family, marriage, children, adoption (ZGB Buch 2)
    "Erbrecht",                   # inheritance, succession (ZGB Buch 3)
    "Sachenrecht",                # property, land registry, mortgage (ZGB Buch 4, GBV)
    "Vertragsrecht",              # obligations, contract law (OR AT + BT)
    "Gesellschaftsrecht",         # company, partnership, merger (OR Buch 2, FusG)
    "Versicherungsvertragsrecht", # private insurance contracts (VVG)
    "Strafrecht",                 # criminal law, juvenile criminal law (StGB, JStG, MStG)
    "Steuerrecht",                # income tax, VAT, withholding, cantonal tax (DBG, MWSTG)
    "Sozialversicherungsrecht",   # AHV, IV, KV, UV, BV, ALV, EL, EO (ATSG umbrella)
    "Arbeitsrecht",               # labor, employment, work safety (OR+ArG+UVG)
    "Finanzmarktrecht",           # banks, securities, insurance supervision (BankG, FIDLEG, FINMAG)
    "Bildungsrecht",              # vocational training, universities, ETH (BBG, HFKG)
    "Gesundheitsrecht",           # health, medicines, epidemics (HMG, EpG, MedBG)
    "Umweltrecht",                # environmental protection, water, air, noise (USG, GSchG)
    "Raumplanungsrecht",          # spatial planning, construction, land use (RPG, NHG)
    "Landwirtschaftsrecht",       # agriculture, food safety, animal welfare (LwG, LMG, TSchG)
    "Forstwirtschaft",            # forestry, hunting, fishing (WaG)
    "Energierecht",               # energy, electricity, nuclear (EnG, StromVG, EleG)
    "Verkehrsrecht",              # road traffic, vehicles (SVG, VTS, VRV)
    "Transportrecht",             # railways, public transport, aviation (EBG, PBG, LFG)
    "Telekommunikationsrecht",    # telecom, post, radio (FMG, PostG, RTVG)
    "Immaterialgüterrecht",       # copyright, trademark, patent, design (URG, MSchG, PatG)
    "Wettbewerbsrecht",           # competition, unfair practices, cartels (KG, UWG)
    "Zollrecht",                  # customs, economic supply (ZG, LVG)
    "Militärrecht",               # military, civil protection, national defense (MG)
]

DOMAINS_BLOCK = '\n'.join(f'  {i+1:2d}. {d}' for i, d in enumerate(DOMAIN_LIST))
print(f'{len(DOMAIN_LIST)} semantic domains defined')
print(DOMAINS_BLOCK)

## Section 3 — Load Gemma & Define Classification Functions

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = 'google/gemma-2-9b-it'
print(f'Loading {MODEL_NAME}...')

quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=quant_cfg, device_map='auto')
model.eval()
print('Model ready')


def call_llm(prompt):
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                             temperature=None, top_p=None,
                             pad_token_id=tokenizer.eos_token_id)
    new_ids = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

In [ ]:
DOMAIN_SET = set(DOMAIN_LIST)

# ── Prompt builders ────────────────────────────────────────────────────────────

def build_law_batch_prompt(batch):
    """batch: list of (law_id, short_title, art1_text)"""
    items = []
    for i, (law_id, title, art1) in enumerate(batch):
        snippet = (title[:100] + '. ' + art1[:300]).strip()
        items.append(f'{i+1}. [{law_id}] {snippet}')
    items_block = '\n'.join(items)
    return f"""You are a Swiss federal law expert building a legal retrieval taxonomy.

Classify each law below into exactly ONE domain from this list:
{DOMAINS_BLOCK}

Laws to classify:
{items_block}

Reply with a JSON object mapping each law_id to its domain name.
Use ONLY domain names from the list above. No explanation.
Example: {{"OR": "Vertragsrecht", "StGB": "Strafrecht"}}"""


def build_court_batch_prompt(batch):
    """batch: list of (case_id, text_snippet)"""
    items = []
    for i, (case_id, snippet) in enumerate(batch):
        items.append(f'{i+1}. [{case_id}] {snippet[:400]}')
    items_block = '\n'.join(items)
    return f"""You are a Swiss federal law expert.

Classify each BGE court decision below into exactly ONE legal domain from this list:
{DOMAINS_BLOCK}

Court decisions to classify:
{items_block}

Reply with a JSON object mapping each case_id to its domain name.
Use ONLY domain names from the list above. No explanation.
Example: {{"BGE 139 I 2": "Verfassungsrecht", "BGE 121 III 5": "Vertragsrecht"}}"""


# ── LLM call + JSON parse ──────────────────────────────────────────────────────

def classify_batch(prompt, expected_keys):
    """Call LLM and parse JSON dict. Returns {id: domain} with 'unknown' fallback."""
    raw = call_llm(prompt)
    # strip markdown fences
    raw = re.sub(r'^```(?:json)?\s*', '', raw.strip())
    raw = re.sub(r'\s*```$', '', raw.strip())
    m = re.search(r'\{[^{}]*\}', raw, re.DOTALL)
    if not m:
        return {k: 'unknown' for k in expected_keys}
    try:
        parsed = json.loads(m.group())
    except json.JSONDecodeError:
        return {k: 'unknown' for k in expected_keys}
    # validate domain names
    result = {}
    for k in expected_keys:
        domain = parsed.get(k, 'unknown')
        result[k] = domain if domain in DOMAIN_SET else 'unknown'
    return result


print('Classification functions ready')

## Section 4 — Classify All Laws

In [ ]:
BATCH_SIZE = 8   # laws per LLM call — balance prompt length vs speed

rows = list(law_meta[['law_id', 'short_title', 'art1_text']].itertuples(index=False))
law_domain = {}  # law_id → domain

for batch_start in range(0, len(rows), BATCH_SIZE):
    batch = rows[batch_start:batch_start + BATCH_SIZE]
    batch_tuples = [(r.law_id, r.short_title, r.art1_text) for r in batch]
    expected_keys = [r.law_id for r in batch]

    prompt = build_law_batch_prompt(batch_tuples)

    for attempt in range(3):
        result = classify_batch(prompt, expected_keys)
        # retry if too many unknowns
        n_unknown = sum(1 for v in result.values() if v == 'unknown')
        if n_unknown <= len(batch) // 2:
            break
        if attempt < 2:
            time.sleep(1)

    law_domain.update(result)

    if (batch_start // BATCH_SIZE) % 20 == 0:
        done = batch_start + len(batch)
        print(f'  [{done}/{len(rows)}] last batch: {result}')

print(f'\nDone. Classified {len(law_domain)} laws.')
unknown_count = sum(1 for v in law_domain.values() if v == 'unknown')
print(f'  unknown: {unknown_count} ({unknown_count/len(law_domain)*100:.1f}%)')

## Section 5 — Classify BGE Court Decisions

In [ ]:
# Extract one representative text per BGE case
# Citation format: "BGE 139 I 2 E. 5.1" → case_id = "BGE 139 I 2"
court['case_id'] = court['citation'].str.extract(r'(BGE\s+\d+\s+[IVX]+\s+\d+)')

# Take the first (lowest-numbered) consideration per case as the domain signal
# Sort by citation to get E. 1 before E. 5 etc.
court_sorted = court.sort_values('citation')
case_rep = (
    court_sorted
    .dropna(subset=['case_id'])
    .drop_duplicates('case_id')
    [['case_id', 'text']]
    .reset_index(drop=True)
)

print(f'Unique BGE cases: {len(case_rep)}')
print(case_rep.head(3))

In [ ]:
BATCH_SIZE_COURT = 5   # BGE texts are longer, use smaller batches

case_rows = list(case_rep[['case_id', 'text']].itertuples(index=False))
court_domain = {}  # case_id → domain

for batch_start in range(0, len(case_rows), BATCH_SIZE_COURT):
    batch = case_rows[batch_start:batch_start + BATCH_SIZE_COURT]
    batch_tuples = [(r.case_id, r.text) for r in batch]
    expected_keys = [r.case_id for r in batch]

    prompt = build_court_batch_prompt(batch_tuples)

    for attempt in range(3):
        result = classify_batch(prompt, expected_keys)
        n_unknown = sum(1 for v in result.values() if v == 'unknown')
        if n_unknown <= len(batch) // 2:
            break
        if attempt < 2:
            time.sleep(1)

    court_domain.update(result)

    if (batch_start // BATCH_SIZE_COURT) % 50 == 0:
        done = batch_start + len(batch)
        print(f'  [{done}/{len(case_rows)}] last: {result}')

print(f'\nDone. Classified {len(court_domain)} BGE cases.')
unknown_court = sum(1 for v in court_domain.values() if v == 'unknown')
print(f'  unknown: {unknown_court} ({unknown_court/len(court_domain)*100:.1f}%)')

## Section 6 — Validate Against Gold Citations

In [ ]:
# Add domain column to every article row
laws['domain'] = laws['law_id'].map(law_domain).fillna('unknown')

# Add domain to every court consideration row
court['domain'] = court['case_id'].map(court_domain).fillna('unknown')

# Gold citation coverage
gold_flat = train['gold_citations'].str.split(';').explode().str.strip()
gold_citations = gold_flat.dropna().tolist()

# Build citation→domain for laws
citation_domain = dict(zip(laws['citation'], laws['domain']))

# Build citation→domain for court (match on case_id prefix)
court_citation_domain = {}
for _, row in court.iterrows():
    if pd.notna(row['case_id']):
        court_citation_domain[row['citation']] = row['domain']

all_citation_domain = {**citation_domain, **court_citation_domain}

# Check coverage
covered    = sum(1 for c in gold_citations if all_citation_domain.get(c, 'unknown') != 'unknown')
unknown_g  = sum(1 for c in gold_citations if all_citation_domain.get(c, 'unknown') == 'unknown')
not_found  = sum(1 for c in gold_citations if c not in all_citation_domain)

print(f'Gold citations: {len(gold_citations)}')
print(f'  domain assigned: {covered}  ({covered/len(gold_citations)*100:.1f}%)')
print(f'  unknown domain:  {unknown_g}')
print(f'  not in corpus:   {not_found}')
print()

# Domain distribution of gold citations
domain_counts = pd.Series([all_citation_domain.get(c, 'unknown') for c in gold_citations]).value_counts()
print('Gold citation domain distribution:')
print(domain_counts.to_string())

## Section 7 — Save Mappings

In [ ]:
# 1. law_domain_mapping.csv — law_id → domain
law_map_df = pd.DataFrame([
    {'law_id': law_id, 'domain': domain}
    for law_id, domain in law_domain.items()
])
law_map_df.to_csv('law_domain_mapping.csv', index=False)
print(f'Saved: law_domain_mapping.csv  ({len(law_map_df)} laws)')

# 2. court_domain_mapping.csv — case_id → domain
court_map_df = pd.DataFrame([
    {'case_id': case_id, 'domain': domain}
    for case_id, domain in court_domain.items()
])
court_map_df.to_csv('court_domain_mapping.csv', index=False)
print(f'Saved: court_domain_mapping.csv  ({len(court_map_df)} BGE cases)')

# 3. articles_with_domain.csv — every article row with domain label
laws[['citation', 'law_id', 'domain']].to_csv('articles_with_domain.csv', index=False)
print(f'Saved: articles_with_domain.csv  ({len(laws):,} articles)')

# 4. court_with_domain.csv — every consideration row with domain label
court[['citation', 'case_id', 'domain']].to_csv('court_with_domain.csv', index=False)
print(f'Saved: court_with_domain.csv  ({len(court):,} considerations)')

# 5. domain_list.json — the taxonomy vocabulary
with open('domain_list.json', 'w') as f:
    json.dump(DOMAIN_LIST, f, ensure_ascii=False, indent=2)
print('Saved: domain_list.json')

In [ ]:
# Domain size summary — how many articles and cases per domain
law_domain_counts  = laws['domain'].value_counts()
court_domain_counts = court['domain'].value_counts()

summary = pd.DataFrame({
    'law_articles':    law_domain_counts,
    'court_cases':     court_domain_counts,
}).fillna(0).astype(int)
summary['total'] = summary['law_articles'] + summary['court_cases']
summary = summary.sort_values('total', ascending=False)

print('Domain sizes (articles + court considerations):')
print(summary.to_string())
print()
print(f'Total: {summary["total"].sum():,} indexed items across {len(summary)} domains')
print(f'Avg per domain: {summary["total"].mean():.0f}')